In [ ]:
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor

# 1. Load Data (force a fresh copy)
df = pd.read_csv("../data/processed/DI_Leather_preprocessed.csv")

# 2. Apply Permissible Feature Engineering (No working_hours allowed here)
df['machine_worker_ratio'] = df['machine_count'] / df['no_of_workers']

# 3. Define Features and Target
target = 'working_hours'

# Safely drop all columns that shouldn't be in this specific model, 
# including any engineered columns that might accidentally be in memory.
drop_cols = [
    'planned_qty', 'production_gap_qty', 'bottleneck_flag', 
    'machine_capacity_per_day', 'week_start_date', 'week_end_date', 
    'financial_year', 'theoretical_max_production', 'avg_hours_per_worker'
]

# Drop columns, using errors='ignore' in case they are already gone
X = df.drop(columns=[target] + drop_cols, errors='ignore')
y = df[target]

# 4. Define Preprocessor
categorical_ohe = ['financial_month', 'factory_unit', 'product_family', 'grade', 'order_category', 'size_id']
categorical_target = ['item_name']
numerical_cols = [col for col in X.columns if col not in categorical_ohe + categorical_target]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat_ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_ohe),
        ('cat_te', TargetEncoder(target_type='continuous'), categorical_target) 
    ])

# 5. Build and Train the Pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), 
    ('model', GradientBoostingRegressor(random_state=42))
])

print("Training Resource Planning model (Predicting working_hours)...")
pipeline.fit(X, y)

# 6. Save the Pipeline
model_filename = 'resource_planning_model_hours.joblib'
joblib.dump(pipeline, model_filename)
print(f"Model successfully saved to {model_filename}")

Training Resource Planning model (Predicting working_hours)...
Model successfully saved to resource_planning_model_hours.joblib
